# Local: Preprocess One Part

Set `PART_ID`, `LOCAL_WORK_ROOT`, or `PACKAGE_TAR` in the first code cell, then run all cells on the local Python/Jupyter environment.

In [ ]:
from __future__ import annotations

import os
import tarfile
import time
from math import gcd
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from datasets import Dataset
from scipy.signal import resample_poly
from sklearn.model_selection import train_test_split
from transformers import AutoProcessor


def log(*values) -> None:
    print(*values, flush=True)


def configure_ssl_cert() -> None:
    """
    Windows terminals sometimes inherit a broken SSL_CERT_FILE path.
    Hugging Face/httpx then fails before downloading the processor.
    """
    try:
        import certifi
    except ImportError:
        return

    cert_path = certifi.where()
    os.environ["SSL_CERT_FILE"] = cert_path
    os.environ["REQUESTS_CA_BUNDLE"] = cert_path
    log("SSL cert:", cert_path)


# ============================================================
# Local-side config
# ============================================================

# Local workers normally use PART_ID 4,5,6,7.
PART_ID = int(os.environ.get("PART_ID", "0"))
NUM_PARTS = 8

# whisper-small uses 80 Mel bins. If this is changed to large-v3 or turbo,
# AutoProcessor automatically uses the model's 128-Mel configuration.
MODEL_ID = os.environ.get("MODEL_ID", "openai/whisper-small")

# Put downloaded server package tar files here.
# Default "." means the current terminal folder.
LOCAL_WORK_ROOT = Path(os.environ.get("LOCAL_WORK_ROOT", ".")).resolve()

PACKAGE_TAR = Path(
    os.environ.get(
        "PACKAGE_TAR",
        str(LOCAL_WORK_ROOT / f"aihub186_whisper_common_part{PART_ID}.tar"),
    )
)

EXTRACT_ROOT = LOCAL_WORK_ROOT / "extracted"
PACKAGE_DIR = EXTRACT_ROOT / f"aihub186_part{PART_ID}"
ASR_TSV = PACKAGE_DIR / f"asr_dataset_part{PART_ID}.tsv"

OUT_ROOT = LOCAL_WORK_ROOT / "processed"
COMMON_OUT_NAME = "hf_dataset_whisper_common_8part"
OUT_DS = OUT_ROOT / f"{COMMON_OUT_NAME}_part{PART_ID}"

# This file is what you upload back to the server.
RESULT_TAR = LOCAL_WORK_ROOT / f"{COMMON_OUT_NAME}_part{PART_ID}.tar"

TARGET_SAMPLE_RATE = 16000
NUM_PROC = 1
MIN_DURATION_SEC = 0.1
MAX_DURATION_SEC = 30.0
SEED = 42

# Loaded once in preprocess_and_save(). Dataset.map calls prepare_batch().
processor = None


def extract_package() -> None:
    if PACKAGE_DIR.is_dir() and ASR_TSV.is_file():
        log("package already extracted:", PACKAGE_DIR)
        return

    if not PACKAGE_TAR.is_file():
        raise FileNotFoundError(
            f"Missing package tar: {PACKAGE_TAR}\n"
            "Download it from the server first."
        )

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    log("extracting:", PACKAGE_TAR)
    with tarfile.open(PACKAGE_TAR, mode="r") as tar:
        tar.extractall(EXTRACT_ROOT)
    log("extracted to:", PACKAGE_DIR)


def load_part_tsv() -> pd.DataFrame:
    if not ASR_TSV.is_file():
        raise FileNotFoundError(ASR_TSV)

    df = pd.read_csv(
        ASR_TSV,
        sep="\t",
        encoding="utf-8-sig",
        dtype={
            "file_id": str,
            "audio_path": str,
            "transcript": str,
            "dataset_type": str,
        },
    )

    df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()

    required_columns = {
        "file_id",
        "audio_path",
        "transcript",
        "duration_sec",
        "dataset_type",
    }
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing TSV columns: {sorted(missing)}")

    df = df.copy()
    df["file_id"] = df["file_id"].fillna("").astype(str).str.strip()
    df["audio_path"] = df["audio_path"].fillna("").astype(str).str.strip()
    df["transcript"] = df["transcript"].fillna("").astype(str).str.strip()
    df["dataset_type"] = (
        df["dataset_type"].fillna("").astype(str).str.strip().str.lower()
    )
    df["duration_sec"] = pd.to_numeric(df["duration_sec"], errors="coerce")

    df["audio_path"] = df["audio_path"].map(
        lambda value: str((PACKAGE_DIR / value).resolve())
    )

    df = df[
        df["file_id"].ne("")
        & df["audio_path"].ne("")
        & df["transcript"].ne("")
        & df["duration_sec"].notna()
        & df["duration_sec"].gt(0)
        & df["dataset_type"].isin(["train", "validation"])
    ].copy()

    df["audio_exists"] = df["audio_path"].map(lambda value: Path(value).is_file())
    missing_audio = df[~df["audio_exists"]]
    if not missing_audio.empty:
        raise FileNotFoundError(
            "Missing local audio files. first="
            + str(missing_audio[["file_id", "audio_path"]].head(5).to_dict("records"))
        )

    df = df[df["audio_exists"]].drop(columns=["audio_exists"])
    df = df[
        df["duration_sec"].between(
            MIN_DURATION_SEC,
            MAX_DURATION_SEC,
            inclusive="both",
        )
    ].copy()

    df = df.drop_duplicates(
        subset=["file_id", "audio_path", "transcript"]
    ).reset_index(drop=True)

    log("rows:", len(df))
    log(df["dataset_type"].value_counts())
    return df


def split_dataframe(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_full_df = df[df["dataset_type"].eq("train")].copy()
    validation_df = df[df["dataset_type"].eq("validation")].copy()

    if train_full_df.empty:
        raise ValueError("No train rows")
    if validation_df.empty:
        raise ValueError("No validation rows")
    if len(validation_df) >= len(train_full_df):
        raise ValueError(
            "validation rows must be fewer than train rows: "
            f"train={len(train_full_df)}, validation={len(validation_df)}"
        )

    development_ratio = len(validation_df) / len(train_full_df)
    train_df, development_df = train_test_split(
        train_full_df,
        test_size=development_ratio,
        random_state=SEED,
        shuffle=True,
    )

    log("train:", len(train_df))
    log("development:", len(development_df))
    log("validation:", len(validation_df))
    return train_df, development_df, validation_df


def to_hf_dataset(source_df: pd.DataFrame) -> Dataset:
    return Dataset.from_dict(
        {
            "file_id": source_df["file_id"].astype(str).tolist(),
            "audio_path": source_df["audio_path"].astype(str).tolist(),
            "text": source_df["transcript"].astype(str).tolist(),
            "duration_sec": source_df["duration_sec"].astype(float).tolist(),
        }
    )


def resample_audio(
    audio_array: np.ndarray,
    original_sample_rate: int,
    target_sample_rate: int,
) -> np.ndarray:
    """Resample mono audio with polyphase anti-alias filtering."""
    if original_sample_rate <= 0:
        raise ValueError(f"Invalid sample rate: {original_sample_rate}")

    if original_sample_rate == target_sample_rate:
        return np.asarray(audio_array, dtype=np.float32)

    common_divisor = gcd(original_sample_rate, target_sample_rate)
    up = target_sample_rate // common_divisor
    down = original_sample_rate // common_divisor

    resampled = resample_poly(audio_array, up=up, down=down)
    return np.asarray(resampled, dtype=np.float32)


def prepare_batch(batch: dict) -> dict:
    if processor is None:
        raise RuntimeError("Processor has not been loaded")

    wav_path = batch["audio_path"]

    audio_array, sample_rate = sf.read(
        wav_path,
        dtype="float32",
        always_2d=False,
    )
    audio_array = np.asarray(audio_array, dtype=np.float32)

    if audio_array.size == 0:
        raise ValueError(f"Empty audio: {wav_path}")

    # soundfile returns multi-channel audio as (frames, channels).
    if audio_array.ndim > 1:
        audio_array = audio_array.mean(axis=1, dtype=np.float32)

    if not np.isfinite(audio_array).all():
        raise ValueError(f"Non-finite audio values: {wav_path}")

    if sample_rate != TARGET_SAMPLE_RATE:
        audio_array = resample_audio(
            audio_array,
            original_sample_rate=sample_rate,
            target_sample_rate=TARGET_SAMPLE_RATE,
        )
        sample_rate = TARGET_SAMPLE_RATE

    if audio_array.size == 0:
        raise ValueError(f"Empty audio after resampling: {wav_path}")
    if not np.isfinite(audio_array).all():
        raise ValueError(f"Non-finite values after resampling: {wav_path}")

    batch["input_features"] = processor.feature_extractor(
        audio_array,
        sampling_rate=sample_rate,
    ).input_features[0]

    batch["labels"] = processor.tokenizer(
        batch["text"],
        add_special_tokens=True,
    ).input_ids

    return batch


def preprocess_and_save() -> None:
    if OUT_DS.exists():
        raise FileExistsError(f"Output dataset already exists: {OUT_DS}")

    df = load_part_tsv()
    train_df, development_df, validation_df = split_dataframe(df)

    ds_train = to_hf_dataset(train_df)
    ds_development = to_hf_dataset(development_df)
    ds_validation = to_hf_dataset(validation_df)

    remove_columns = ["audio_path", "text"]
    map_kwargs = {
        "remove_columns": remove_columns,
    }
    if NUM_PROC and NUM_PROC > 1:
        map_kwargs["num_proc"] = NUM_PROC

    log("loading processor:", MODEL_ID)
    global processor
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        language="ko",
        task="transcribe",
    )

    processor_sample_rate = int(processor.feature_extractor.sampling_rate)
    if processor_sample_rate != TARGET_SAMPLE_RATE:
        raise ValueError(
            "TARGET_SAMPLE_RATE does not match the processor: "
            f"target={TARGET_SAMPLE_RATE}, processor={processor_sample_rate}"
        )

    log("processor sample rate:", processor_sample_rate)
    log("processor n_mels:", processor.feature_extractor.feature_size)
    log("processor chunk length:", processor.feature_extractor.chunk_length)

    started = time.time()
    log("preprocessing train split...")
    ds_train_prep = ds_train.map(
        prepare_batch,
        **map_kwargs,
        desc="Preparing ASR train dataset",
    )
    log("train done sec:", round(time.time() - started, 1))

    started = time.time()
    log("preprocessing development split...")
    ds_development_prep = ds_development.map(
        prepare_batch,
        **map_kwargs,
        desc="Preparing ASR development dataset",
    )
    log("development done sec:", round(time.time() - started, 1))

    started = time.time()
    log("preprocessing validation split...")
    ds_validation_prep = ds_validation.map(
        prepare_batch,
        **map_kwargs,
        desc="Preparing ASR validation dataset",
    )
    log("validation done sec:", round(time.time() - started, 1))

    OUT_DS.mkdir(parents=True, exist_ok=False)
    log("saving train:", OUT_DS / "train")
    ds_train_prep.save_to_disk(str(OUT_DS / "train"))
    log("saving development:", OUT_DS / "development")
    ds_development_prep.save_to_disk(str(OUT_DS / "development"))
    log("saving validation:", OUT_DS / "validation")
    ds_validation_prep.save_to_disk(str(OUT_DS / "validation"))

    log("saved:", OUT_DS)


def create_result_tar() -> None:
    if not OUT_DS.is_dir():
        raise FileNotFoundError(OUT_DS)
    if RESULT_TAR.exists():
        raise FileExistsError(f"Result tar already exists: {RESULT_TAR}")

    RESULT_TAR.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(RESULT_TAR, mode="w") as tar:
        tar.add(OUT_DS, arcname=OUT_DS.name)

    log("result tar:", RESULT_TAR)
    log("\nUpload this file back to the server, for example:")
    log(
        "rsync -avh --partial --progress "
        f"{RESULT_TAR} "
        "USER@TARGET_SERVER:/home/data/expr/week2/04-asr_aihub/uploaded_local_parts/"
    )


def main() -> None:
    if PART_ID < 0 or PART_ID >= NUM_PARTS:
        raise ValueError(f"PART_ID must be 0..{NUM_PARTS - 1}: {PART_ID}")

    log("PART_ID:", PART_ID, "/", NUM_PARTS)
    log("MODEL_ID:", MODEL_ID)
    log("PACKAGE_TAR:", PACKAGE_TAR)
    log("OUT_DS:", OUT_DS)
    log("RESULT_TAR:", RESULT_TAR)

    configure_ssl_cert()
    extract_package()
    preprocess_and_save()
    create_result_tar()


if __name__ == "__main__":
    import multiprocessing

    multiprocessing.freeze_support()
    main()


PART_ID: 0 / 8
PACKAGE_TAR: /Users/leeminhyeok/Desktop/summer_bootcamp/minhyeok/aihub186_whisper_common_part0.tar
OUT_DS: /Users/leeminhyeok/Desktop/summer_bootcamp/minhyeok/processed/hf_dataset_whisper_common_8part_part0
RESULT_TAR: /Users/leeminhyeok/Desktop/summer_bootcamp/minhyeok/hf_dataset_whisper_common_8part_part0.tar
SSL cert: /opt/anaconda3/envs/llm1/lib/python3.11/site-packages/certifi/cacert.pem
extracting: /Users/leeminhyeok/Desktop/summer_bootcamp/minhyeok/aihub186_whisper_common_part0.tar
extracted to: /Users/leeminhyeok/Desktop/summer_bootcamp/minhyeok/extracted/aihub186_part0
rows: 21360
dataset_type
train         19137
validation     2223
Name: count, dtype: int64
train: 16914
development: 2223
validation: 2223
loading processor: openai/whisper-small


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessing train split...


Preparing ASR train dataset:   0%|          | 0/16914 [00:00<?, ? examples/s]

ValueError: Expected 16000Hz, got 44100: file_id=HOS13000825042A029, path=/Users/leeminhyeok/Desktop/summer_bootcamp/minhyeok/extracted/aihub186_part0/audio/HOS13000825042A029.wav